# Notebook 05 — RAG Setup (LangChain + ChromaDB + Groq)

**Stack:** LangChain · ChromaDB · Groq (llama-3.3-70b) · HuggingFace Embeddings

**Before running:** Make sure `.env` exists with `GROQ_API_KEY=your_key`

In [1]:
# CELL 1 — Install dependencies
!pip install langchain langchain-groq langchain-chroma langchain-community \
             langchain-huggingface chromadb sentence-transformers \
             python-dotenv -q
             
print('✅ Dependencies installed')

✅ Dependencies installed


In [5]:
# CELL 2 — Imports and config
import os
from pathlib import Path
from dotenv import load_dotenv

from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

load_dotenv()

# Adjust paths if running from notebooks/ subfolder
POLICIES_DIR = Path('../policies')
CHROMA_DIR   = Path('../backend/models/chroma_db')
GROQ_API_KEY = os.environ.get('GROQ_API_KEY', '')

print(f'Policies dir : {POLICIES_DIR.resolve()}')
print(f'ChromaDB dir : {CHROMA_DIR.resolve()}')
print(f'Groq key set : {"✅ Yes" if GROQ_API_KEY else "❌ No — set GROQ_API_KEY in .env"}')

Policies dir : /home/user/Documents/credit-card-dispute-resolution/policies
ChromaDB dir : /home/user/Documents/credit-card-dispute-resolution/backend/models/chroma_db
Groq key set : ✅ Yes


In [6]:
# CELL 3 — Load and inspect policy files
loader = DirectoryLoader(
    str(POLICIES_DIR),
    glob='**/*.md',
    loader_cls=TextLoader,
    loader_kwargs={'encoding': 'utf-8'},
)
raw_docs = loader.load()

print(f'✅ Loaded {len(raw_docs)} policy files:')
for doc in raw_docs:
    words = len(doc.page_content.split())
    fname = Path(doc.metadata['source']).name
    print(f'   {fname:<40} {words:>4} words')

print(f'\nSample content from first file:')
print(raw_docs[0].page_content[:400])

✅ Loaded 6 policy files:
   billing_error.md                          860 words
   duplicate_charge.md                       863 words
   service_not_provided.md                  1157 words
   goods_not_received.md                     972 words
   unauthorized_transaction.md               702 words
   merchant_fraud.md                        1066 words

Sample content from first file:
# Billing Error Policy

## Definition
A billing error is any incorrect charge that appears on a cardholder's credit card statement where the amount billed differs from the amount agreed upon, authorized, or expected. Unlike unauthorized transactions (where the cardholder did not authorize any charge), billing errors involve legitimate transactions where the amount, description, or terms are incorr


In [7]:
# CELL 4 — Chunk the documents
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=80,
    separators=['\n\n', '\n', '. ', ' '],
)

chunks = splitter.split_documents(raw_docs)

# Enrich metadata with category from filename
for chunk in chunks:
    source = Path(chunk.metadata['source']).stem
    chunk.metadata['category'] = source.replace('_', ' ').title()

print(f'✅ Split into {len(chunks)} chunks')
print(f'   Avg chunk size: {sum(len(c.page_content) for c in chunks) // len(chunks)} chars')

from collections import Counter
counts = Counter(c.metadata['category'] for c in chunks)
print(f'\nChunks per policy:')
for cat, count in sorted(counts.items()):
    print(f'   {cat:<35} {count} chunks')

print(f'\nSample chunk:')
print(chunks[0].page_content)

✅ Split into 112 chunks
   Avg chunk size: 333 chars

Chunks per policy:
   Billing Error                       17 chunks
   Duplicate Charge                    17 chunks
   Goods Not Received                  17 chunks
   Merchant Fraud                      23 chunks
   Service Not Provided                24 chunks
   Unauthorized Transaction            14 chunks

Sample chunk:
# Billing Error Policy


In [8]:
# CELL 5 — Create ChromaDB vector store (run once — persists to disk)
CHROMA_DIR.mkdir(parents=True, exist_ok=True)

# Free local embedding model — no API key needed
embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2',
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True},
)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=str(CHROMA_DIR),
    collection_name='policy_docs',
)

print(f'✅ ChromaDB index created and saved!')
print(f'   Path   : {CHROMA_DIR.resolve()}')
print(f'   Vectors: {vectorstore._collection.count()}')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1322.24it/s]


✅ ChromaDB index created and saved!
   Path   : /home/user/Documents/credit-card-dispute-resolution/backend/models/chroma_db
   Vectors: 112


In [9]:
# CELL 6 — Test retrieval (no LLM yet — just vector search)
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 3},
)

test_queries = [
    'unauthorized credit card charge I did not make',
    'charged twice for same purchase duplicate transaction',
    'ordered item never received goods not delivered',
    'wrong amount billed overcharged on statement',
    'subscription cancelled but still being charged',
    'fake merchant scam fraudulent website',
]

print('=== RETRIEVAL TESTS ===\n')
for query in test_queries:
    print(f'Query: "{query}"')
    results = retriever.invoke(query)
    for i, doc in enumerate(results, 1):
        cat     = doc.metadata.get('category', 'Unknown')
        snippet = doc.page_content[:100].replace('\n', ' ')
        print(f'  [{i}] [{cat}] {snippet}...')
    print()

=== RETRIEVAL TESTS ===

Query: "unauthorized credit card charge I did not make"
  [1] [Unauthorized Transaction] # Unauthorized Transaction Policy  ## Definition An unauthorized transaction occurs when a credit ca...
  [2] [Unauthorized Transaction] ## Common Scenarios and Outcomes - **Card physically stolen and used:** Chargeback upheld in 95% of ...
  [3] [Unauthorized Transaction] - **Family member used card without permission:** Cardholder must submit a signed declaration. Charg...

Query: "charged twice for same purchase duplicate transaction"
  [1] [Duplicate Charge] ## Definition A duplicate charge occurs when a cardholder is charged more than once for the same sin...
  [2] [Duplicate Charge] ## Qualifying Conditions A dispute qualifies as a duplicate charge when: - The same transaction amou...
  [3] [Duplicate Charge] ## Common Causes of Duplicate Charges 1. **Point-of-sale terminal errors:** Card reader malfunction ...

Query: "ordered item never received goods not delivered"

In [10]:
# CELL 7 — Connect to Groq LLM
llm = ChatGroq(
    model='llama-3.3-70b-versatile',  # best free model on Groq (fast + smart)
    temperature=0.2,                   # low = factual and consistent
    max_tokens=512,
    api_key=GROQ_API_KEY,
)

# Quick connectivity test
response = llm.invoke('Say only: Groq connected successfully.')
print(f'✅ {response.content}')

✅ Groq connected successfully.


In [15]:
# CELL 8 — Define RAG prompt and chain builder
DISPUTE_PROMPT = ChatPromptTemplate.from_template("""
You are a professional bank dispute resolution officer.

A customer submitted this complaint:
"{complaint}"

Our NLP classifier predicted the dispute category as: {category}

Relevant bank policy (use this as your only reference):
{context}

Write a professional response in exactly 3 sentences:
1. Explain why this complaint falls under "{category}"
2. Describe what the resolution process involves  
3. Tell the customer what to expect (timeline and likely outcome)

Be factual, concise, and professional. No bullet points or headers.
""")

def format_docs(docs):
    return '\n\n'.join(doc.page_content for doc in docs)


from operator import itemgetter
def build_rag_chain(category: str):
    """Build a retrieval chain filtered to a specific dispute category."""
    cat_retriever = vectorstore.as_retriever(
        search_type='similarity',
        search_kwargs={
            'k': 3,
            'filter': {'category': category},
        },
    )
    return (
        {
            'context':   itemgetter('complaint') | cat_retriever | format_docs,
            'complaint': itemgetter('complaint'),
            'category':  itemgetter('category'),
        }
        | DISPUTE_PROMPT
        | llm
        | StrOutputParser()
    )

print('✅ RAG chain builder ready')

✅ RAG chain builder ready


In [16]:
# CELL 9 — Full pipeline test (all 6 categories)
test_cases = [
    ('Someone made a $500 purchase on my credit card without my permission. I never authorized this transaction.', 'Unauthorized Transaction'),
    ('I was charged twice for the same $89.99 grocery purchase on the same day from the same merchant.', 'Duplicate Charge'),
    ('I ordered a laptop 3 months ago and it never arrived. The seller stopped responding to my emails.', 'Goods Not Received'),
    ('I was billed $150 but the merchant and I agreed on $50. This is a clear billing error on my statement.', 'Billing Error'),
    ('I cancelled my gym subscription in January but they kept charging my card every month for 4 more months.', 'Service Not Provided'),
    ('The website I ordered from was completely fake. I paid $300 but the company never existed. I was scammed.', 'Merchant Fraud'),
]

print('=== FULL RAG PIPELINE TEST ===\n')
for complaint, category in test_cases:
    print(f'Category   : {category}')
    print(f'Complaint  : {complaint[:80]}...')
    chain = build_rag_chain(category)
    explanation = chain.invoke({'complaint': complaint, 'category': category})
    print(f'Explanation: {explanation}')
    print('-' * 65)
    print()

=== FULL RAG PIPELINE TEST ===

Category   : Unauthorized Transaction
Complaint  : Someone made a $500 purchase on my credit card without my permission. I never au...


Explanation: This complaint falls under our Unauthorized Transaction policy as the customer explicitly states that they did not authorize the $500 purchase made on their credit card, indicating a transaction was made without their knowledge or consent. The resolution process involves verifying the details of the disputed transaction, reviewing the customer's account activity, and potentially initiating a chargeback process to reverse the unauthorized charge, which may require additional verification if a family member was involved. The customer can expect a thorough investigation to be completed within 7-10 business days, and if the transaction is confirmed as unauthorized, the disputed amount will likely be credited back to their account, with the possibility of their card being replaced if necessary to prevent further unauthorized activity.
-----------------------------------------------------------------

Category   : Duplicate Charge
Complaint  : I was charged twice for the same $8

In [ ]:
# CELL 10 — Verify ChromaDB persists and reloads correctly
vectorstore_reloaded = Chroma(
    persist_directory=str(CHROMA_DIR),
    embedding_function=embeddings,
    collection_name='policy_docs',
)

count = vectorstore_reloaded._collection.count()
print(f'✅ ChromaDB reloaded successfully')
print(f'   Vectors : {count}')
print(f'   Path    : {CHROMA_DIR.resolve()}')
print()

✅ ChromaDB reloaded successfully
   Vectors : 112
   Path    : /home/user/Documents/credit-card-dispute-resolution/backend/models/chroma_db

Your chroma_db/ is saved at backend/models/chroma_db/
The FastAPI backend will load it automatically at startup.

Next step: build rag_engine.py and update main.py
